[Guide Index](README.md)


# 05d The SHACL Library Registry

`starlayer.starontology` bundles five ontology/SHACL-shapes pairs, each describing a vocabulary this stack uses internally:

| Prefix | Pair | What it is |
|---|---|---|
| `manch:` | `manchester_owl` / `manchester_shacl` | OWL 2 Manchester Syntax's own internal AST as RDF |
| `skos:` | `skos_owl` / `skos_shacl` | The W3C SKOS Reference's formal axioms and integrity conditions, over the real `skos:` namespace |
| `salg:` | `sparql_owl` / `sparql_shacl` | A SPARQL 1.2 query/update's own algebra as RDF |
| `sqe:` | `sqe_owl` / `sqe_shacl` | A SPARQL query's raw parse tree (pre-`translateQuery`), for hand/UI editing |
| `srl:` | `srl_owl` / `srl_shacl` | SRL's own rule-set abstract syntax as RDF |

Plus an eleventh entry, `shacl_meta` — the SHACL 1.2 meta-shapes themselves (shapes-about-shapes), generated rather than file-backed, so it has no ontology counterpart.

Every entry lives as its own named graph inside one bundled dataset, `starontology.trig` - one file to look at, one file to edit, regardless of which sibling package (`starlayer.graph`, `starlayer.sparql`, `starlayer.shacl`) ends up consuming a given entry. Real, standalone per-entry `.ttl` files also exist alongside it (`starlayer/starontology/*.ttl`) - but those are a *published* artifact regenerated from the dataset (`scripts/publish_starontology_files.py`), purely so each entry has a real, fetchable URL (section 6 below) - edit `starontology.trig`, never these.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later steps reuse variables from earlier ones.


In [1]:
from starlayer import starontology

## 1. List what's registered

`get_ontology_list()` returns every entry's metadata (`name`, `description`, `type`), sorted by name — no `pyshacl` import needed for this, just reading the registry.

In [2]:
for info in starontology.get_ontology_list():
    print(f"{info.name:20} {info.type:10} {info.description[:80]}")

manchester_owl       ast-graph  OWL 2 Manchester Syntax's own internal AST as RDF (manch:) - frames/clauses/clas
manchester_shacl     shacl      SHACL shapes validating a manch: AST graph's own structural well-formedness.
shacl_meta           shacl      The SHACL 1.2 meta-shapes themselves (starlayer.shacl's own shapes-about-shapes,
skos_owl             owl        The W3C SKOS Reference's own formal axioms and numbered Documented Consistency a
skos_shacl           shacl      SHACL shapes checking SKOS's own numbered integrity conditions against a data gr
sparql_owl           ast-graph  A SPARQL 1.2 query/update's own algebra as RDF (salg:) - encode/edit/decode/re-e
sparql_shacl         shacl      SHACL shapes validating a salg: algebra graph's own structural well-formedness.
spexpr_owl           ast-graph  The SPARQL expression/variable/property-path vocabulary (spexpr:), carved out of
spexpr_shacl         shacl      SHACL shapes validating an spexpr: expression/variable/path value's o

## 2. Load an entry's graph

`get_ontology_graph(name)` returns a fresh `StarLayerGraph` of the named file — a real, standalone artifact usable by any RDF tool, not something the registry itself invents. Raises `KeyError` (naming every valid choice) for a typo'd name, rather than returning `None`.

In [3]:
skos_owl = starontology.get_ontology_graph("skos_owl")
skos_shacl = starontology.get_ontology_graph("skos_shacl")
print("skos_owl:  ", len(skos_owl), "triples")
print("skos_shacl:", len(skos_shacl), "triples")

skos_owl:   90 triples
skos_shacl: 102 triples


## 3. Validate real data against a registered shapes graph

Any of the five `*_shacl` entries works directly with `StarShaclSchema`, the same as a hand-authored shapes graph.

In [4]:
from starlayer import StarLayerGraph, StarShaclSchema

data = StarLayerGraph()
data.parse(data="""
    @prefix skos: <http://www.w3.org/2004/02/skos/core#> .
    @prefix ex: <http://example.org/> .
    ex:scheme1 a skos:ConceptScheme .
    ex:concept1 a skos:Concept ; skos:inScheme ex:scheme1 ; skos:prefLabel "Example"@en .
""", format="turtle")

result = StarShaclSchema(shacl_graph=skos_shacl).validate(data_graph=data, meta_shacl=False)
print("conforms:", result.conforms)

conforms: True


## 4. Every shapes graph is a profile of its own ontology

This is the SHACL 1.2 Profiling spec's own `prof:isProfileOf` relationship (`dx-prof`, `http://www.w3.org/ns/dx/prof/` — see the [SHACL Profiling guide](04e-shacl-profiling.ipynb) for the full background), applied uniformly across every pair in this registry: whenever an ontology/AST graph has a paired shapes graph validating it, that shapes graph is declared a *profile of* the ontology — regardless of whether the ontology itself also happens to formalize some further external standard (true for `skos:`, not for `srl:`).

This mapping lives in a real, persisted third document — `ontology-shapes-profiles.ttl` — deliberately separate from both the ontology/AST graphs it relates (often formalizing something external, like the real `skos:` namespace) and the shapes graphs themselves (functional validation rules, not self-description). `get_conformance_profiles(url=None)` loads it — the bundled copy by default, or any local path/`http(s)://` location instead. `get_profile_graph(shapes_name, url=None)` is a thin filter over that same document for one pair; `get_resource_iri(name)` returns the matching resource IRI for either side.

In [5]:
shapes_names = ["manchester_shacl", "skos_shacl", "sparql_shacl", "sqe_shacl", "srl_shacl"]

for name in shapes_names:
    profile_graph = starontology.get_profile_graph(name)
    for s, p, o in profile_graph:
        print(f"{name:18} -> {o}")

# shacl_meta has no ontology counterpart in this registry - None, not an error
print("shacl_meta:", starontology.get_profile_graph("shacl_meta"))

# the whole mapping document at once, instead of one pair at a time
print()
print("whole mapping document:", len(starontology.get_conformance_profiles()), "triples")

manchester_shacl   -> http://www.w3.org/ns/dx/prof/role/validation
manchester_shacl   -> http://www.w3.org/ns/dx/prof/ResourceDescriptor
manchester_shacl   -> https://github.com/hidden-graph/starlayer/blob/main/packages/starlayer/starlayer/starontology/manchester_shapes.ttl
manchester_shacl   -> http://www.w3.org/ns/dx/prof/ResourceDescriptor
manchester_shacl   -> SHACL shapes validating a manch: AST graph's own structural well-formedness.
manchester_shacl   -> http://www.w3.org/ns/dx/prof/role/schema
manchester_shacl   -> https://github.com/hidden-graph/starlayer/blob/main/packages/starlayer/starlayer/starontology/manchester-ast-ontology.ttl
manchester_shacl   -> OWL 2 Manchester Syntax's own internal AST as RDF (manch:) - frames/clauses/class- and property-expressions/data-ranges, one level upstream of the compiled OWL RDF mapping.
manchester_shacl   -> n6fce387ea13d4e9e94b7a82c7d797977b4
manchester_shacl   -> n6fce387ea13d4e9e94b7a82c7d797977b5
manchester_shacl   -> OWL 2 Manchester

## 5. The full conformance chain

`get_resource_iri()` gives a validated dataset's own `shapes_graph_resource=` the *same* identity `get_profile_graph()` asserts `prof:isProfileOf` on — so `shacl_profiling.derive_conforms_to()`'s second rule can actually match it and derive a `dcterms:conformsTo` triple all the way through to the ontology, not just `sh:conformsTo` to the shapes graph.

In [6]:
from starlayer.shacl import shacl_profiling

result = StarShaclSchema(shacl_graph=skos_shacl).validate(
    data_graph=data,
    meta_shacl=False,
    data_graph_iri="http://example.org/graphs/my-skos-data",
    shapes_graph_resource=starontology.get_resource_iri("skos_shacl"),
)

# conforms_only=True: just the sh:conformsTo/dcterms:conformsTo triples
# themselves - not report_graph merged in too (derive_conforms_to()'s
# own default), and not even the sh:validationReport link it otherwise
# adds. Keeps this demo to exactly the conformance claims being derived.
derived = shacl_profiling.derive_conforms_to(
    result.report_graph,
    specification_graph=starontology.get_profile_graph("skos_shacl"),
    conforms_only=True,
)
for row in derived:
    print(row)

(rdflib.term.URIRef('http://example.org/graphs/my-skos-data'), rdflib.term.URIRef('http://www.w3.org/ns/shacl#conformsTo'), rdflib.term.URIRef('https://github.com/hidden-graph/starlayer/blob/main/packages/starlayer/starlayer/starontology/skos_shapes.ttl'))
(rdflib.term.URIRef('http://example.org/graphs/my-skos-data'), rdflib.term.URIRef('http://purl.org/dc/terms/conformsTo'), rdflib.term.URIRef('https://www.w3.org/TR/skos-reference'))


## 6. Fetching an entry directly from GitHub, outside Python

`get_resource_iri(name)` (section 5 above) is this registry's own stable *identity* for an entry - a GitHub blob URL. Nice to browse, but a plain HTTP GET against a blob URL returns GitHub's own HTML viewer page, not the file's real bytes. `get_raw_url(name)` is a different thing entirely: a `raw.githubusercontent.com` URL that returns the entry's real content - fetchable by anything, independent of StarLayer being installed at all.


In [ ]:
print("identity (get_resource_iri):", starontology.get_resource_iri("skos_shacl"))
print("fetch   (get_raw_url):     ", starontology.get_raw_url("skos_shacl"))


Real content, fetched live over the network - no local file, no bundled package data involved on the reading end:


In [ ]:
remote = StarLayerGraph()
remote.parse(starontology.get_raw_url("skos_owl"), format="turtle")
print(len(remote), "triples fetched directly from GitHub")


`get_raw_url()` also covers `"ontology_shapes_profiles"` (the mapping document from section 4) - it has no `get_resource_iri()` counterpart at all, since it isn't one of the registry's own ontology/shapes entries:


In [ ]:
print(starontology.get_raw_url("ontology_shapes_profiles"))


The three SHACL asset files `starlayer.shacl` itself bundles (the meta-shapes' own presentation/validation rules, and the SHACL 1.2 conformance self-declaration from the [SHACL profiling guide](04e-shacl-profiling.ipynb)) get the identical treatment from a sibling module, `starlayer.shacl.asset_urls` - already real, standalone files, so no publish step is needed for these three, just URL-minting:


In [ ]:
from starlayer.shacl import asset_urls

for name in ("presentation_shapes", "validation_shapes", "conformance_profile"):
    print(f"{name:20} {asset_urls.get_raw_url(name)}")
